# Semantic Chunk Pruning

TF-IDF based semantic similarity ranking for context compression.

## How It Works

1. **Chunk the document** into overlapping windows (default: 180 words, 40 word overlap)
2. **Vectorize** both the question and all chunks using TF-IDF with bigrams
3. **Compute cosine similarity** between the question vector and each chunk vector
4. **Rank chunks** by similarity score and keep the top-k most relevant

## Trade-offs

| Pros | Cons |
|------|------|
| Captures semantic relevance better than regex | Requires scikit-learn dependency |
| Handles synonyms/partial matches via n-grams | TF-IDF is less nuanced than neural embeddings |
| Overlapping chunks preserve local context | Chunk size is a hyperparameter to tune |

## Section A: Setup and Imports

In [ ]:
import numpy as np
from typing import List
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Section B: Core Helper Functions

In [ ]:
def chunk_text(text: str, chunk_size_words: int = 180, overlap_words: int = 40) -> List[str]:
    """
    Split text into overlapping chunks.

    Args:
        text: Input text
        chunk_size_words: Words per chunk
        overlap_words: Overlap between consecutive chunks

    Returns:
        List of text chunks
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    step = max(1, chunk_size_words - overlap_words)
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size_words]
        if chunk:
            chunks.append(' '.join(chunk))
    return chunks

## Section C: Semantic Chunk Pruning Method

In [ ]:
def semantic_chunk_prune(text: str, question: str, top_k: int = 8) -> str:
    """
    Prune context by ranking chunks via TF-IDF cosine similarity to the question.

    Args:
        text: The full document text
        question: The user's query
        top_k: Number of top chunks to keep

    Returns:
        Pruned text with most semantically relevant chunks
    """
    chunks = chunk_text(text)
    if not chunks:
        return ''

    # Build TF-IDF vectors for question and all chunks
    vect = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    vect.fit([question] + chunks)

    q_vec = vect.transform([question])
    c_vec = vect.transform(chunks)

    # Rank by cosine similarity and keep top-k
    scores = cosine_similarity(q_vec, c_vec).flatten()
    idx = np.argsort(scores)[::-1][:min(top_k, len(chunks))]

    selected = [chunks[i] for i in idx]
    return '\n\n'.join(selected)

## Section D: Demo and Testing

In [ ]:
sample_text = (
    "The company reported strong Q3 earnings. Revenue grew by 15% year over year. "
    "The expansion of the Frankfurt server farm cost $14.73 million. This was partially "
    "offset by a $2.1 million green energy tax rebate. Employee headcount remained stable. "
    "Management raised full-year guidance citing strong demand in European markets."
)

sample_question = "What is the projected infrastructure capital expenditure for Q3 2026?"

print(f"Original length: {len(sample_text.split())} words")
print(f"Number of chunks: {len(chunk_text(sample_text))}")
print("\n" + "="*60)

result = semantic_chunk_prune(sample_text, sample_question, top_k=3)
print(f"\nPruned output:\n{result}")